# U-01: Forecast uncertainty quantification (FC-03, D-40)

Calibrated prediction intervals for the production forecasters, three paradigms:
- **Split-conformal** (model-agnostic): RF (hourly) + DLinear (daily), calibrated on 2021, tested 2022–23, **Mondrian per horizon**.
- **Quantile XGBoost** (`reg:quantileerror`, α=0.05/0.5/0.95) — both tracks.
- **LSTM-pinball** (quantile head + pinball loss) — both tracks, GPU.

Metrics (Towers 4/9, per horizon): **PICP@90%** (coverage; target 0.90), **MPIW** (interval width / sharpness), **pinball loss**.
Expected: intervals wide (episodic CH₄ spikes = high aleatoric uncertainty); coverage near/just-below 0.90 (2022–23 vs 2021 shift).

In [1]:
import sys, datetime; sys.path.insert(0, "../../src/models")
import numpy as np, pandas as pd, torch, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import forecasting_dl as fdl
from pathlib import Path
RESULTS=Path("../../results"); FIG=RESULTS/"figures"
dev=fdl.get_device(); print("device",dev)
m=fdl.load_matrix(); AR=[c for c in m.columns if c.startswith("ar_")]; FX=fdl.FX; DUM=["is_t2","is_t4","is_t9"]; FEAT=AR+FX+DUM
QS=[0.05,0.5,0.95]; HA=[1,6,12,24,48]; HB=[1,3,7,14]
def conf_level(n,a=0.1): return min(1.0,np.ceil((n+1)*(1-a))/max(n,1))
def picp(y,lo,hi): return float(((y>=lo)&(y<=hi)).mean()) if len(y) else np.nan
def mpiw(lo,hi): return float(np.mean(hi-lo)) if len(lo) else np.nan
def pinball(y,qp,qs): return float(np.mean([np.mean(np.maximum(a*(y-qp[:,i]),(a-1)*(y-qp[:,i]))) for i,a in enumerate(qs)])) if len(y) else np.nan
def aligned_A(h):
    parts=[]
    for t in [2,4,9]:
        df=m[m.tower==t].set_index("Datetime").sort_index(); f=df[AR+DUM].copy()
        for c in FX: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["tower"]=t; f["ttime"]=df.index+pd.Timedelta(hours=h); parts.append(f)
    return pd.concat(parts)
ARB=[f"ar_ch4_dlag{L}" for L in [1,2,3,7,14]]+["ar_ch4_drm7"]
def daily_tab():
    T={}
    for t in [2,4,9]:
        df=m[m.tower==t].set_index("Datetime").sort_index()
        cnt=df["y_observed"].resample("D").count(); yo=df["y_observed"].resample("D").mean().where(cnt>=6)
        yg=df["y_gapfilled"].resample("D").mean(); dd=pd.DataFrame(index=yg.index); dd["y_observed"]=yo
        for L in [1,2,3,7,14]: dd[f"ar_ch4_dlag{L}"]=yg.shift(L)
        dd["ar_ch4_drm7"]=yg.shift(1).rolling(7,min_periods=1).mean()
        for c in FX: dd[c]=df[c].resample("D").mean()
        for tt in [2,4,9]: dd[f"is_t{tt}"]=1.0 if tt==t else 0.0
        dd["tower"]=t; T[t]=dd
    return T
TB=daily_tab(); FEATB=ARB+FX+DUM
def aligned_B(h):
    parts=[]
    for t in [2,4,9]:
        df=TB[t]; f=df[ARB+DUM].copy()
        for c in FX: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["tower"]=t; f["ttime"]=df.index+pd.Timedelta(days=h); parts.append(f)
    return pd.concat(parts)
print("helpers ready")

device cuda


helpers ready


## 1  Split-conformal — RF (hourly) + DLinear (daily)

In [2]:
ROWS=[]
# Conformal RF (Track A)
for h in HA:
    A=aligned_A(h); imp=SimpleImputer(strategy="mean")
    ptr=A[A.ttime.dt.year.between(2018,2020)&A.target.notna()]; cal=A[(A.ttime.dt.year==2021)&A.target.notna()]
    rf=RandomForestRegressor(200,min_samples_leaf=5,n_jobs=-1,random_state=42); rf.fit(imp.fit_transform(ptr[FEAT].values),ptr["target"].values)
    for t in [4,9]:
        c=cal[cal.tower==t]; rc=np.abs(c["target"].values-rf.predict(imp.transform(c[FEAT].values)))
        q=np.quantile(rc,conf_level(len(rc)))
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]; y=te["target"].values; yp=rf.predict(imp.transform(te[FEAT].values))
        ROWS.append(dict(method="conformal",model="RF",track="A",tower=t,horizon=h,PICP=round(picp(y,yp-q,yp+q),3),MPIW=round(mpiw(yp-q,yp+q),1),pinball=np.nan,n=len(y)))
print("conformal RF done")
# Conformal DLinear (Track B) via DL harness
WB=fdl.build_windows(m,"B"); cut20=pd.Timestamp("2020-12-31 23:59")
proper=[fdl._subset(WB[t],pd.DatetimeIndex(WB[t]["ttime"][:,-1])<=cut20) for t in [2,4,9]]; train=fdl._cat(proper)
cal={t:fdl._subset(WB[t],pd.DatetimeIndex(WB[t]["otime"]).year==2021) for t in [4,9]}
test={t:fdl._subset(WB[t],pd.DatetimeIndex(WB[t]["otime"]).year.isin([2022,2023])) for t in [4,9]}
mu,sd=fdl._standardize(train,[cal[4],cal[9],test[4],test[9]]); ne,nd=WB[4]["enc"].shape[-1],WB[4]["dec"].shape[-1]
dl=fdl.build_model("DLinear",28,14,ne,nd,3); fdl.train_model(dl,train,dev,epochs=40,ch4_mu=mu,ch4_sd=sd)
for t in [4,9]:
    pc=fdl.predict(dl,cal[t],dev,mu,sd); pt=fdl.predict(dl,test[t],dev,mu,sd)
    for h in HB:
        k=h-1; yc=cal[t]["y"][:,k]; okc=np.isfinite(yc); rc=np.abs(yc[okc]-pc[okc,k])
        if len(rc)<5: continue
        q=np.quantile(rc,conf_level(len(rc))); yt=test[t]["y"][:,k]; okt=np.isfinite(yt); y=yt[okt]; p=pt[okt,k]
        ROWS.append(dict(method="conformal",model="DLinear",track="B",tower=t,horizon=h,PICP=round(picp(y,p-q,p+q),3),MPIW=round(mpiw(p-q,p+q),1),pinball=np.nan,n=len(y)))
print("conformal DLinear done")

conformal RF done


conformal DLinear done


## 2  Quantile XGBoost (both tracks)

In [3]:
def qxgb(builder, feat, h, hor_unit):
    A=builder(h); imp=SimpleImputer(strategy="mean")
    tr=A[A.ttime.dt.year.between(2018,2021)&A.target.notna()]; Xtr=imp.fit_transform(tr[feat].values)
    out=[]
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        Xte=imp.transform(te[feat].values); y=te["target"].values; QP=np.zeros((len(te),3))
        for i,a in enumerate(QS):
            xg=XGBRegressor(objective="reg:quantileerror",quantile_alpha=a,n_estimators=300,max_depth=5,learning_rate=0.05,subsample=0.8,n_jobs=-1)
            xg.fit(Xtr,tr["target"].values); QP[:,i]=xg.predict(Xte)
        QP=np.sort(QP,axis=1)
        out.append(dict(method="quantile",model="XGB",track=hor_unit,tower=t,horizon=h,PICP=round(picp(y,QP[:,0],QP[:,2]),3),MPIW=round(mpiw(QP[:,0],QP[:,2]),1),pinball=round(pinball(y,QP,QS),2),n=len(y)))
    return out
for h in HA: ROWS+=qxgb(aligned_A,FEAT,h,"A")
for h in HB: ROWS+=qxgb(aligned_B,FEATB,h,"B")
print("quantile-XGB done")

quantile-XGB done


## 3  LSTM-pinball (both tracks, GPU)

In [4]:
def lstm_pinball(track, HSET):
    Wt=fdl.build_windows(m,track); cfg=fdl.TRACKS[track]; cut=pd.Timestamp("2021-12-31 23:59")
    trp=[fdl._subset(Wt[t],pd.DatetimeIndex(Wt[t]["ttime"][:,-1])<=cut) for t in [2,4,9]]; train=fdl._cat(trp)
    test={t:fdl._subset(Wt[t],pd.DatetimeIndex(Wt[t]["otime"]).year.isin([2022,2023])) for t in [4,9]}
    mu,sd=fdl._standardize(train,[test[4],test[9]]); ne,nd=Wt[4]["enc"].shape[-1],Wt[4]["dec"].shape[-1]
    qm=fdl.LSTMQuantile(ne,nd,3,cfg["H"],3); fdl.train_quantile(qm,train,dev,epochs=30,ch4_mu=mu,ch4_sd=sd)
    out=[]
    for t in [4,9]:
        QL=fdl.predict_quantile(qm,test[t],dev,mu,sd)
        for h in HSET:
            k=h-1; yk=test[t]["y"][:,k]; ok=np.isfinite(yk); y=yk[ok]
            if len(y)<10: continue
            q0,q1,q2=QL[ok,k,0],QL[ok,k,1],QL[ok,k,2]
            out.append(dict(method="quantile",model="LSTM",track=track,tower=t,horizon=h,PICP=round(picp(y,q0,q2),3),MPIW=round(mpiw(q0,q2),1),pinball=round(pinball(y,np.stack([q0,q1,q2],1),QS),2),n=len(y)))
    return out
ROWS+=lstm_pinball("A",HA); ROWS+=lstm_pinball("B",HB)
R=pd.DataFrame(ROWS); print("LSTM-pinball done | total UQ rows",len(R))

LSTM-pinball done | total UQ rows 54


## 4  Results, figures, logging

In [5]:
print("=== PICP@90% (coverage; target 0.90) — Track A, Tower 4 ===")
print(R[(R.track=='A')&(R.tower==4)].pivot_table(index=['method','model'],columns='horizon',values='PICP').round(3).to_string())
print("\n=== MPIW (interval width) — Track A, Tower 4 ===")
print(R[(R.track=='A')&(R.tower==4)].pivot_table(index=['method','model'],columns='horizon',values='MPIW').round(0).to_string())
print("\n=== pinball loss (quantile models) — Track A, Tower 4 ===")
print(R[(R.track=='A')&(R.tower==4)&(R.pinball.notna())].pivot_table(index='model',columns='horizon',values='pinball').round(2).to_string())
# Fig 1: PICP vs horizon (Track A, T4)
fig,ax=plt.subplots(1,2,figsize=(13,4.2))
sa=R[(R.track=='A')&(R.tower==4)]
for (meth,mdl),g in sa.groupby(['method','model']):
    g=g.sort_values('horizon'); ax[0].plot(g.horizon,g.PICP,'o-',label=f"{meth}-{mdl}"); ax[1].plot(g.horizon,g.MPIW,'o-',label=f"{meth}-{mdl}")
ax[0].axhline(0.9,ls='--',c='k',label='target 0.90'); ax[0].set_title('Coverage (PICP) vs horizon — T4 hourly'); ax[0].set_xlabel('h'); ax[0].set_ylabel('PICP'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].set_title('Sharpness (MPIW) vs horizon — T4 hourly'); ax[1].set_xlabel('h'); ax[1].set_ylabel('interval width'); ax[1].grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG/'fc03_coverage_sharpness.png',dpi=130); plt.close(fig)
# Fig 2: fan chart — conformal-RF, T4, h=24, summer 2022
h=24; A=aligned_A(h); imp=SimpleImputer(strategy='mean')
ptr=A[A.ttime.dt.year.between(2018,2020)&A.target.notna()]; cal=A[(A.ttime.dt.year==2021)&A.target.notna()]
rf=RandomForestRegressor(200,min_samples_leaf=5,n_jobs=-1,random_state=42); rf.fit(imp.fit_transform(ptr[FEAT].values),ptr['target'].values)
c=cal[cal.tower==4]; q=np.quantile(np.abs(c['target'].values-rf.predict(imp.transform(c[FEAT].values))),conf_level(len(c)))
te=A[(A.tower==4)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()].copy(); te['pred']=rf.predict(imp.transform(te[FEAT].values)); te=te.set_index('ttime').sort_index()
win=te['2022-06-15':'2022-07-10']
fig,ax=plt.subplots(figsize=(13,4))
ax.fill_between(win.index,win['pred']-q,win['pred']+q,alpha=.25,color='#1f77b4',label='90% conformal PI')
ax.plot(win.index,win['pred'],color='#1f77b4',lw=1.2,label='RF forecast (+24h)'); ax.plot(win.index,win['target'],'k.',ms=4,label='observed')
ax.set_title('Conformal 90% interval — Tower 4, 24h-ahead (summer 2022)'); ax.set_ylabel('FCH4 (nmol m⁻² s⁻¹)'); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(FIG/'fc03_fanchart_T4_h24.png',dpi=130); plt.close(fig)
# log
R.to_csv(RESULTS/'fc03_uq_summary.csv',index=False)
bench=RESULTS/'benchmarks.csv'; today=datetime.date.today().isoformat(); ex=pd.read_csv(bench); ex=ex[ex['replication']!='FC-03']
br=[]
for _,r in R.iterrows():
    br.append({'replication':'FC-03','model':f"{r['method']}_{r['model']}",'tower':f"Tower {int(r.tower)}",'track':r['track'],'horizon':int(r['horizon']),
        'feature_set':'UQ 90% interval','picp':r['PICP'],'mpiw':r['MPIW'],'pinball':r['pinball'],'n_test':int(r['n']),'date':today,
        'notes':'FC-03 uncertainty quantification (conformal/quantile-XGB/LSTM-pinball, D-40)'})
comb=pd.concat([ex,pd.DataFrame(br)],ignore_index=True); comb.to_csv(bench,index=False)
print(f"\nWrote {len(br)} FC-03 rows. Total {len(comb)}. Saved fc03_uq_summary.csv + 2 figures")

=== PICP@90% (coverage; target 0.90) — Track A, Tower 4 ===
horizon             1      6      12     24     48
method    model                                   
conformal RF     0.886  0.888  0.884  0.883  0.885
quantile  LSTM   0.818  0.763  0.737  0.736  0.731
          XGB    0.861  0.865  0.860  0.863  0.856

=== MPIW (interval width) — Track A, Tower 4 ===
horizon             1      6      12     24     48
method    model                                   
conformal RF     227.0  261.0  237.0  253.0  258.0
quantile  LSTM   201.0  190.0  189.0  185.0  184.0
          XGB    166.0  174.0  177.0  180.0  180.0

=== pinball loss (quantile models) — Track A, Tower 4 ===
horizon     1      6      12     24     48
model                                     
LSTM     16.14  19.05  19.87  20.13  20.17
XGB      14.83  16.04  15.89  15.95  16.37



Wrote 54 FC-03 rows. Total 3098. Saved fc03_uq_summary.csv + 2 figures
